# Exploratory Data Analysis (EDA)
# Wisconsin Breast Cancer Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
%matplotlib inline
plt.style.use('default')
sns.set_palette("husl")

# Import custom modules
import sys
sys.path.append('../src')
from data_loader import load_data, get_dataset_info
from preprocessing import analyze_class_imbalance

## 1. Load Dataset

In [ ]:
# Load breast cancer dataset using sklearn
X, y = load_data(source="sklearn")

print(f"Dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

## 2. Dataset Information

In [ ]:
# Display dataset info
info = get_dataset_info()
print("Dataset Information:")
for key, value in info.items():
    print(f"{key}: {value}")

print(f"\nData types:\n{X.dtypes.value_counts()}")
print(f"\nMissing values: {X.isnull().sum().sum()}")

## 3. Target Distribution

In [ ]:
# Analyze class distribution
class_analysis = analyze_class_imbalance(y)

# Create bar chart for target distribution
plt.figure(figsize=(8, 5))
counts = y.value_counts().sort_index()
labels = ['Benign (0)', 'Malignant (1)']
colors = ['lightblue', 'salmon']

bars = plt.bar(labels, counts.values, color=colors, alpha=0.7, edgecolor='black')
plt.title('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count')

# Add value labels on bars
for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Class imbalance ratio: {class_analysis['imbalance_ratio']:.2f}")
print(f"Minority class percentage: {class_analysis['class_percentages'][0]:.1f}%")

**Observation**: The dataset shows moderate class imbalance with ~37% benign and ~63% malignant cases. This imbalance should be addressed during preprocessing.

## 4. Summary Statistics

In [ ]:
# Display summary statistics
print("Summary Statistics:")
summary_stats = X.describe()
print(summary_stats.round(3))

**Observation**: Features have very different scales (e.g., area ranges from 143-2501 while smoothness ranges from 0.05-0.16). Scaling will be essential for most algorithms.

## 5. Correlation Analysis

In [ ]:
# Calculate correlation with target
target_corr = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 features most correlated with target:")
print(target_corr.head(10).round(3))

# Get the 6 most correlated features for detailed analysis
top_6_features = target_corr.head(6).index.tolist()
print(f"\nTop 6 features: {top_6_features}")

In [ ]:
# Create correlation heatmap
plt.figure(figsize=(20, 16))
correlation_matrix = X.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm', 
            center=0, square=True, linewidths=0.1, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation**: Strong correlations exist between related features (e.g., radius, perimeter, area). This multicollinearity suggests feature selection or dimensionality reduction may be beneficial.

## 6. Distribution of Most Correlated Features

In [ ]:
# Plot histograms for the 6 most correlated features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(top_6_features):
    # Plot histogram for each class
    benign_data = X[y == 0][feature]
    malignant_data = X[y == 1][feature]
    
    axes[i].hist(benign_data, alpha=0.7, label='Benign', color='lightblue', bins=30)
    axes[i].hist(malignant_data, alpha=0.7, label='Malignant', color='salmon', bins=30)
    
    axes[i].set_title(f'{feature}\n(corr: {target_corr[feature]:.3f})', fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observation**: Clear separation between benign and malignant cases is visible in most features, with malignant tumors generally showing higher values for size and texture-related features.

## 7. Boxplots by Class

In [ ]:
# Create boxplots for top features split by class
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Combine data for easier plotting
data_combined = X.copy()
data_combined['diagnosis'] = y.map({0: 'Benign', 1: 'Malignant'})

for i, feature in enumerate(top_6_features):
    sns.boxplot(data=data_combined, x='diagnosis', y=feature, ax=axes[i])
    axes[i].set_title(f'{feature}', fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observation**: Boxplots confirm significant differences between classes and reveal outliers in several features. Outlier handling may improve model performance.

## 8. Feature Group Analysis

In [ ]:
# Analyze correlations by feature groups (mean, error, worst)
mean_features = [col for col in X.columns if col.startswith('mean')]
error_features = [col for col in X.columns if 'error' in col]
worst_features = [col for col in X.columns if col.startswith('worst')]

print("Average correlation with target by feature group:")
print(f"Mean features: {X[mean_features].corrwith(y).abs().mean():.3f}")
print(f"Error features: {X[error_features].corrwith(y).abs().mean():.3f}")
print(f"Worst features: {X[worst_features].corrwith(y).abs().mean():.3f}")

# Plot correlation comparison
group_corrs = {
    'Mean': X[mean_features].corrwith(y).abs().mean(),
    'Error': X[error_features].corrwith(y).abs().mean(),
    'Worst': X[worst_features].corrwith(y).abs().mean()
}

plt.figure(figsize=(8, 5))
bars = plt.bar(group_corrs.keys(), group_corrs.values(), 
               color=['skyblue', 'lightgreen', 'coral'], alpha=0.7, edgecolor='black')
plt.title('Average Correlation with Target by Feature Group', fontweight='bold')
plt.ylabel('Average Absolute Correlation')

# Add value labels
for bar, value in zip(bars, group_corrs.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

**Observation**: 'Worst' features show the highest correlation with the target, followed by 'mean' features. 'Error' features are least predictive, suggesting they could be candidates for removal during feature selection.

## 9. Statistical Significance Testing

In [ ]:
# Perform t-tests for each feature
p_values = []
effect_sizes = []

for feature in X.columns:
    benign = X[y == 0][feature]
    malignant = X[y == 1][feature]
    
    # T-test
    t_stat, p_val = stats.ttest_ind(benign, malignant)
    p_values.append(p_val)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt(((len(benign) - 1) * benign.var() + 
                         (len(malignant) - 1) * malignant.var()) / 
                        (len(benign) + len(malignant) - 2))
    cohens_d = abs(benign.mean() - malignant.mean()) / pooled_std
    effect_sizes.append(cohens_d)

# Create results dataframe
significance_results = pd.DataFrame({
    'feature': X.columns,
    'p_value': p_values,
    'effect_size': effect_sizes,
    'correlation': X.corrwith(y).abs()
})

# Sort by effect size
significance_results = significance_results.sort_values('effect_size', ascending=False)

print("Top 10 features by effect size:")
print(significance_results.head(10)[['feature', 'effect_size', 'p_value', 'correlation']].round(4))

**Observation**: All features show statistically significant differences between classes (p < 0.05), with large effect sizes for the top features, confirming their discriminative power.

## 10. Key Findings and Preprocessing Recommendations

### Key Findings:

1. **Dataset Quality**: 569 samples, 30 features, no missing values - high quality dataset
2. **Class Imbalance**: Moderate imbalance (37% benign, 63% malignant) - SMOTE recommended
3. **Feature Scaling**: Large scale differences between features - scaling essential
4. **Feature Correlations**: Strong multicollinearity present - feature selection beneficial
5. **Discriminative Power**: 'Worst' features most predictive, 'error' features least predictive
6. **Outliers**: Present in several features - robust scaling or outlier removal recommended

### Preprocessing Recommendations:

1. **Scaling**: Use StandardScaler or RobustScaler (due to outliers)
2. **Class Imbalance**: Apply SMOTE during training
3. **Feature Selection**: Consider removing 'error' features or use dimensionality reduction
4. **Outlier Handling**: Use RobustScaler or consider outlier removal
5. **Cross-validation**: Use stratified CV to maintain class balance
6. **Model Selection**: Tree-based models may handle correlations better than linear models